# Stage 01 — Review (T1 + T3 + M1/M2/M3)

For every patent in `matched/` (output of Stage 00b2 — figures already cropped
**and already matched to BRIEF DESCRIPTION lines**, including
`matched_description`/`match_status`/`fig_number` written into
`crops_mapping.csv`):
1. Read each crop's precomputed match result from `crops_mapping.csv` (no
   re-matching here — see 00b2's matching cell, which has the filenames and
   `data/descriptions.csv` available right after cropping)
2. SigLIP zero-shot: T2 (per-figure visual fields), G1 (architecture topType),
   M1 (fuselage/gear/symmetry), M2 (wing config/empennage), M3 (propulsion)
3. PatentSBERTa: T1 dimensions (scope, field, target) — uses the patent's full
   description text from `data/descriptions.csv` directly (a separate use of
   that file from the per-figure matching in step 1, which now lives in 00b2)
4. Export the consolidated `source_patents.xlsx` (every patent's record is
   returned in-memory only — no per-patent JSON/HTML files are written here)

Human review happens in the single master wizard
(`UI_for_taxonomy_caracterization_10.0.html`): load the batch's
`ml_predict_labels_<batch>.xlsx` (exported by this notebook) via its batch
file input, confirm/correct fields, then export. All inference in this
pipeline runs locally (SigLIP + PatentSBERTa, plus an optional local
InternVL2-8B second opinion) — there is no external API call. There is no
per-patent review-HTML generation step in this notebook.

SigLIP and PatentSBERTa are **required** — the notebook will not run without them.

**match_status colour key** (computed in 00b2, read here as-is):
- 🟢 `matched` — filename label found and matched to a description line (confidence 0.95)
- 🔵 `semantic` — filename label not found verbatim; SBERT picked the closest description line
- 🔴 `unmatched` — filename label found but no matching line in description text (confidence 0.0)
- 🟡 `no_label` — no label in filename; positional fallback used (confidence 0.3)
- 🟠 `duplicate` — same figure number claimed by multiple images (confidence 0.9)
- ⚫ `human_required` — no label, and positional fallback unavailable (mixed split-page patent)

Rows with `needs_review: true` are **highlighted with a yellow border**.

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
repo_root = None
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / "src").exists() and (_candidate / "config.yaml").exists():
        repo_root = _candidate
        break
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}. Run from inside Patent-Labelling-Tools.")

for p in [str(repo_root), str(repo_root / "src")]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))
print(f"repo_root : {repo_root}")


In [ ]:
# == Tunable similarity thresholds & ensemble weights ======================
# Central place to tune the matching / duplicate-detection heuristics. Raise a
# threshold to demand a closer match (fewer, higher-confidence hits); lower it
# to catch more (more hits, more false positives).
#
# IMAGE_DUPLICATE_THRESHOLD - SigLIP image-to-image cosine cutoff above which
#   two figure crops are treated as the SAME drawing (a duplicate image reused
#   across patents, e.g. continuations of one aircraft). 0.80-0.90 is sensible.
# TEXT_MATCH_THRESHOLD - PatentSBERTa text-to-text cosine cutoff above which two
#   patents' title/abstract text is treated as near-duplicate. 0.65-0.80 typical.
IMAGE_DUPLICATE_THRESHOLD = 0.85
TEXT_MATCH_THRESHOLD      = 0.70

# Cross-modal ensemble bias used when a SigLIP (visual) and an SBERT (text)
# prediction disagree on the SAME field. >1 favours that modality on close
# calls; 1.0 / 1.0 = pick the raw higher-confidence side (original behaviour).
# Pushed into src.reviewer just before run_stage01() below.
VISUAL_WEIGHT = 1.0   # trust in SigLIP layout/visual predictions
TEXT_WEIGHT   = 1.0   # trust in PatentSBERTa description-text predictions

print(f'Image-duplicate threshold : {IMAGE_DUPLICATE_THRESHOLD}')
print(f'Text-match threshold      : {TEXT_MATCH_THRESHOLD}')
print(f'Ensemble weights          : visual={VISUAL_WEIGHT}, text={TEXT_WEIGHT}')


In [ ]:
# ── GPU selection + VRAM check ─────────────────────────────────────────────────
# Run this before the cell below if you might be using a GPU for something else
# (another notebook, a training run, etc). It reports free VRAM per card and
# lets you pick which GPU(s) the dual-worker pipeline is allowed to use.
#
# GPU_SELECTION:
#   "auto"    -> use every GPU with enough free VRAM (>= MIN_FREE_GB), one worker each
#   "0"       -> force a single worker on GPU 0 only
#   "1"       -> force a single worker on GPU 1 only
#   "0,1"     -> force both GPUs regardless of current free memory (old default behaviour)
GPU_SELECTION = "0,1"
MIN_FREE_GB   = 6.0   # YOLO + EasyOCR + 4-bit Qwen2.5-VL-7B need roughly this much headroom

import torch

def _free_gb(idx: int) -> float:
    free_bytes, _total_bytes = torch.cuda.mem_get_info(idx)
    return free_bytes / 1e9

n_gpu = torch.cuda.device_count()
print(f"GPUs visible: {n_gpu}")
free_per_gpu = {}
for i in range(n_gpu):
    name = torch.cuda.get_device_name(i)
    free = _free_gb(i)
    free_per_gpu[i] = free
    flag = "OK" if free >= MIN_FREE_GB else "LOW"
    print(f"  GPU {i} ({name}): {free:.1f} GB free  [{flag}]")

if GPU_SELECTION == "auto":
    selected_gpu_ids = [str(i) for i in range(n_gpu) if free_per_gpu.get(i, 0) >= MIN_FREE_GB]
    if not selected_gpu_ids:
        best = max(free_per_gpu, key=free_per_gpu.get, default=0)
        selected_gpu_ids = [str(best)]
        print(f"\nNo GPU has >= {MIN_FREE_GB} GB free -- falling back to GPU {best} only "
              f"({free_per_gpu.get(best, 0):.1f} GB free). Expect possible OOM/'unavailable' "
              f"qwen_status rows; consider closing other GPU processes first.")
    else:
        print(f"\nAuto-selected GPU(s): {', '.join(selected_gpu_ids)}")
else:
    selected_gpu_ids = [s.strip() for s in GPU_SELECTION.split(",") if s.strip()]
    print(f"\nForced GPU selection: {', '.join(selected_gpu_ids)}")
    for s in selected_gpu_ids:
        idx = int(s)
        free = free_per_gpu.get(idx, 0)
        if free < MIN_FREE_GB:
            print(f"  WARNING: GPU {idx} only has {free:.1f} GB free (< {MIN_FREE_GB} GB) -- "
                  f"this worker may hit OOM. Close other processes on this GPU if possible.")

print(f"\nWorkers to launch: {len(selected_gpu_ids)} -> GPU(s) {selected_gpu_ids}")


In [ ]:
# -- GPU stale-process diagnostic (run after the GPU-selection cell above) --
# Memory held by a CUDA context is only released when its OWNING PROCESS
# exits -- torch.cuda.empty_cache() in THIS kernel cannot free memory that a
# different process (another notebook, another conda env, a crashed worker)
# is still holding. This cell just reports what nvidia-smi sees so you can
# tell "my own kernel is using this" from "something else left memory stuck"
# before you decide what to kill -- it does NOT kill anything automatically.
import subprocess, os

my_pid = os.getpid()

def _run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()

proc_csv = _run([
    "nvidia-smi", "--query-compute-apps=pid,used_memory,gpu_uuid",
    "--format=csv,noheader,nounits",
])
gpu_csv = _run(["nvidia-smi", "--query-gpu=index,uuid", "--format=csv,noheader"])
uuid_to_idx = {}
for line in gpu_csv.splitlines():
    idx, uuid = [p.strip() for p in line.split(",")]
    uuid_to_idx[uuid] = idx

print(f"This kernel's PID: {my_pid}\n")
print("GPU compute processes:")
stale = []
for line in proc_csv.splitlines():
    if not line.strip():
        continue
    pid_s, mem_s, uuid = [p.strip() for p in line.split(",")]
    pid = int(pid_s)
    gpu_idx = uuid_to_idx.get(uuid, "?")
    is_me = pid == my_pid
    status = _run(["ps", "-o", "stat=", "-p", str(pid)]) or "gone"
    cmd = _run(["ps", "-o", "cmd=", "-p", str(pid)])
    flag = "<- this kernel" if is_me else ("ZOMBIE/leaked" if status.startswith("Z") or not cmd else "")
    print(f"  GPU {gpu_idx}  PID {pid:>7}  {mem_s:>6} MiB  [{status or 'gone':<4}]  {flag}  {cmd[:70]}")
    if flag == "ZOMBIE/leaked":
        stale.append(pid)

if stale:
    print(f"\n{len(stale)} PID(s) look like leaked/zombie GPU memory from a "
          f"previous run: {stale}")
    print("These won't free themselves -- find their PARENT process "
          "(`ps -o ppid= -p <pid>`) and end IT (e.g. restart that other "
          "Jupyter kernel) once you've confirmed it's not work you still need.")
else:
    print("\nNo obviously leaked GPU processes found.")


In [ ]:
import os
from src.config_loader import load_config
cfg = load_config()

# Must run BEFORE any sentence_transformers/open_clip/huggingface_hub import:
# huggingface_hub freezes HF_HUB_CACHE as a constant at first import, so
# setting it afterward is silently ignored and weights re-download to the
# default ~/.cache/huggingface instead of the project models/ folder.
os.environ["HF_HUB_CACHE"] = str(cfg["paths"]["siglip_cache"])
os.environ["HF_HOME"]      = str(cfg["paths"]["siglip_cache"])

from src.cross_modal import load_siglip_model, verify_matches

# Use whichever GPU the "GPU selection + VRAM check" cell above picked
# (selected_gpu_ids) instead of the bare "cuda" string, which always means
# device 0 -- that silently ignored GPU_SELECTION and was the actual cause
# of the earlier OOM (kept loading onto GPU 0 even when GPU 0 was full and
# GPU 1 had plenty of free VRAM).
_selected = globals().get("selected_gpu_ids")
model_device = f"cuda:{_selected[0]}" if _selected else None

# -- PatentSBERTa -- required for T1 dimension classification --------------
# Cached under cfg["paths"]["sbert_cache"] (Patent-Labelling-Tools/models/SBERT)
# so weights stay inside the project instead of ~/.cache/huggingface.
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer("AI-Growth-Lab/PatentSBERTa", cache_folder=str(cfg["paths"]["sbert_cache"]), device=model_device)
print(f"PatentSBERTa ready on {sbert.device}.")

# -- SigLIP -- required for T2/G1/M1/M2/M3 visual classification -----------
USE_SIGLIP = True
siglip_bundle = load_siglip_model(cache_dir=cfg["paths"]["siglip_cache"], device=model_device)
print(f"SigLIP ready on {siglip_bundle[3]}.")


In [ ]:
# ── Batch selector ────────────────────────────────────────────────────────
# Set BATCH_ID to the batch you want to review (see data/batches.xlsx Summary
# sheet — sheets are 1-indexed: Batch_01 .. Batch_05, there is no Batch_00).
BATCH_ID = 5

# Set to a folder name under matched/ to bypass batches.xlsx entirely and test
# against an ad-hoc folder (e.g. a tester batch that has no batches.xlsx sheet).
# Leave as None for a real run driven by BATCH_ID.
OVERRIDE_BATCH_DIR = None   # e.g. "Batch_01_review_tester"

import pandas as pd
from pathlib import Path

cfg = __import__("src.config_loader", fromlist=["load_config"]).load_config()

if OVERRIDE_BATCH_DIR:
    sheet_name = OVERRIDE_BATCH_DIR
    matched_probe = Path(cfg["paths"]["matched"]) / sheet_name
    patent_ids = sorted(p.name for p in matched_probe.iterdir() if p.is_dir())
    batch_df = pd.DataFrame({"patent_id": patent_ids})
    print(f"OVERRIDE: {len(patent_ids)} patents loaded from matched/{sheet_name}/ (no batches.xlsx sheet used)")
else:
    batches_path = Path(cfg["paths"]["data"]) / "batches.xlsx"
    if not batches_path.exists():
        raise FileNotFoundError(f"batches.xlsx not found at {batches_path} — run 00b1_grouping first.")

    sheet_name  = f"Batch_{BATCH_ID:02d}"
    batch_df    = pd.read_excel(batches_path, sheet_name=sheet_name, dtype=str)
    patent_ids  = batch_df["patent_id"].dropna().str.strip().tolist()

    print(f"Batch {BATCH_ID}: {len(patent_ids)} patents loaded from {sheet_name}")
    print(batch_df[["patent_id","company_canonical","prototype_label"]].head(10).to_string(index=False))


In [ ]:
from src.config_loader import load_config
from src.extractor import load_patseer_excel
from src.reviewer import process_patent

cfg = load_config()
# matched/ is now nested per batch (Stage 00b2 writes to matched/Batch_NN/) —
# use the same sheet_name computed in the batch-selector cell above so this
# notebook looks in the same place 00b2 wrote to.
from pathlib import Path as _Path
matched_dir = _Path(cfg['paths']['matched']) / sheet_name
print(f"matched : {matched_dir}")


In [ ]:
# Excel used for T1 metadata only (title, assignee, years, citations).
# Drawing descriptions are read from data/descriptions.csv by process_patent().
excel_index = load_patseer_excel(cfg['paths']['patseer_excel'])
print(f"Indexed {len(excel_index)} patents.")

In [ ]:
# ── Single-patent diagnostic (optional) ─────────────────────────────────────
# Set INSPECT_ID to the bare patent_id (matches excel_index / descriptions.csv
# keys) — NOT the matched/ folder name, which carries a "_{record_number}"
# suffix. process_patent() resolves the folder internally via
# resolve_patent_image_dir(), and returns the assembled record dict directly
# (no JSON file is written here).
# Leave as None to skip.

INSPECT_ID = "EP3770063A1"

if INSPECT_ID:
    from src.reviewer import process_patent
    data = process_patent(
        INSPECT_ID, cfg, excel_index, matched_dir,
        sbert_model=sbert, siglip_bundle=siglip_bundle, skip_siglip=not USE_SIGLIP,
    )
    print(f"Inspected: {INSPECT_ID}")
    print(f"Figures: {len(data.get('T3_images', []))}")
    print(f"has_splits: {data.get('has_splits')}")


In [ ]:
# ── Batch run configuration ──────────────────────────────────────────────────
LIMIT = None  # int or None — None processes all patents in matched/
# SigLIP and SBERT are always on — see model-loading cell above.
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
# ── Skip flagged crops from SigLIP processing (Cell 5c re-crop flag) ───────
# Cell 5c of 00b2 lets a reviewer flag a crop "re_crop" via the [✂ Re-crop]
# button — those crops, and any already "rejected", must be excluded here
# until a human manually re-crops and clears the flag in crops_mapping.csv.
import functools
import pandas as pd
from src import reviewer as _reviewer

SKIP_QUALITIES = {"rejected", "re_crop"}

# crops_mapping.csv CAN be per-batch: data/<Batch_NN>/crops_mapping_<Batch_NN>.csv
# (named after sheet_name, written by 00b2) — but matched/ and data/ don't always
# get nested in lockstep, so fall back to the flat data/crops_mapping.csv when
# the per-batch copy isn't actually on disk, instead of silently excluding nothing.
_batch_crops_csv = Path(cfg["paths"]["data_matched"]) / sheet_name / f"crops_mapping_{sheet_name}.csv"
_flat_crops_csv  = Path(cfg["paths"]["data"]) / "crops_mapping.csv"
_crops_csv = _batch_crops_csv if _batch_crops_csv.exists() else _flat_crops_csv

if _crops_csv.exists():
    _crops_df = pd.read_csv(_crops_csv)
    skip_files = set(
        _crops_df.loc[_crops_df["crop_quality"].isin(SKIP_QUALITIES), "output"].dropna()
    )
else:
    skip_files = set()

print(f"Reading crop quality from: {_crops_csv}")
print(f"Excluding {len(skip_files)} crop(s) from SigLIP processing (qualities: {sorted(SKIP_QUALITIES)}).")

_reviewer.process_patent = functools.partial(_reviewer.process_patent, skip_files=skip_files)


In [ ]:
# Apply the tunable ensemble weights from the thresholds cell so this run's
# SigLIP-vs-SBERT field reconciliation honours them. run_stage01_parallel()
# (selected when selected_gpu_ids has >1 entry) reads these as module globals
# too and forwards them into each GPU worker explicitly, since a worker is a
# fresh process that never sees this notebook's monkey-patch.
import src.reviewer as _rv
_rv.VISUAL_WEIGHT, _rv.TEXT_WEIGHT = VISUAL_WEIGHT, TEXT_WEIGHT

from src.reviewer import run_stage01, run_stage01_parallel

_run_patent_ids = patent_ids[:LIMIT] if LIMIT else patent_ids
print(f"Processing {len(_run_patent_ids)}/{len(patent_ids)} patent(s) from {sheet_name} (LIMIT={LIMIT}).")

# selected_gpu_ids comes from the "GPU selection + VRAM check" cell above.
# >1 GPU selected -> split the patent list across one worker per GPU
# (run_stage01_parallel, src/review_gpu_worker.py) instead of loading one
# model pair and walking the list serially on a single device.
if len(selected_gpu_ids) > 1:
    print(f"Splitting across {len(selected_gpu_ids)} GPU(s): {selected_gpu_ids}")
    summary_df = run_stage01_parallel(
        cfg,
        matched_dir      = matched_dir,    # matched/Batch_NN/ — matches 00b2's per-batch output
        skip_siglip      = not USE_SIGLIP,
        patent_ids       = _run_patent_ids,  # process only this batch (sliced by LIMIT)
        enrich_citations = True,
        gpu_ids          = selected_gpu_ids,
        skip_files        = skip_files,      # from the "skip flagged crops" cell above
    )
else:
    summary_df = run_stage01(
        cfg,
        sbert_model   = sbert,           # loaded in model-loading cell above
        siglip_bundle = siglip_bundle,   # None if SigLIP not loaded
        skip_siglip   = not USE_SIGLIP,
        patent_ids    = _run_patent_ids,   # process only this batch (sliced by LIMIT)
        matched_dir   = matched_dir,  # matched/Batch_NN/ — matches 00b2's per-batch output
    )

In [ ]:
print(summary_df.to_string(index=False))

In [ ]:
# == Duplicate detection (SigLIP image-to-image + PatentSBERTa text-to-text) ==
# Runs AFTER run_stage01 wrote ml_predict_labels_<batch>.xlsx. Two separate
# modalities (never crossed): SigLIP compares IMAGE-to-IMAGE to flag the same
# drawing reused across patents; PatentSBERTa compares TEXT-to-TEXT to flag
# near-duplicate patents. Thresholds come from the top-of-notebook constants.
# Re-runnable: it rewrites the same xlsx in place.
import re
import pandas as pd
from pathlib import Path
from src.cross_modal import detect_image_duplicates, detect_text_duplicates

_data_matched = Path(cfg['paths'].get('data_matched', cfg['paths']['data']))
_labels_xlsx  = _data_matched / sheet_name / f'ml_predict_labels_{sheet_name}.xlsx'
if not _labels_xlsx.exists():
    raise FileNotFoundError(f'{_labels_xlsx} not found - run the run_stage01 cell first.')
_df = pd.read_excel(_labels_xlsx, sheet_name='Review')

# fig_num parsed from 'Image: <filename>', same convention as the HTML wizard's
# figNumFromFilename(): _F<n>.png -> n; bare _Fu.png -> Fu_D<dnum> (unique per crop).
def _fig_num(fname):
    m = re.search(r'_Fu(\d+)\.png$', fname)
    if m: return 'Fu' + m.group(1)
    if re.search(r'_Fu\.png$', fname):
        d = re.search(r'_D(\d+)_', fname)
        return ('Fu_D' + d.group(1)) if d else fname.rsplit('.', 1)[0]
    m = re.search(r'_F(\w+)\.png$', fname)
    if m: return m.group(1)
    return fname.rsplit('.', 1)[0]

# -- Collect one image entry per T2 figure (skip missing files) --
_t2 = _df[_df['Section'] == 'T2']
_entries, _seen = [], set()
for _, r in _t2.iterrows():
    sub = str(r.get('Sub_Dimension') or '')
    if not sub.startswith('Image: '): continue
    fname, path = sub[len('Image: '):], r.get('Image_Path')
    key = (str(r['Patent_ID']), fname)
    if key in _seen or not isinstance(path, str) or not Path(path).exists():
        continue
    _seen.add(key)
    _entries.append({'patent_id': str(r['Patent_ID']), 'fig_num': _fig_num(fname),
                     'filename': fname, 'image_path': path})

print(f'Encoding {len(_entries)} figure crop(s) for image-duplicate detection...')
_model, _tok, _pre, _dev = siglip_bundle
_img_dups = detect_image_duplicates(_entries, _model, _pre, _dev,
                                    threshold=IMAGE_DUPLICATE_THRESHOLD)
print(f'  {len(_img_dups)} duplicate image(s) found (threshold {IMAGE_DUPLICATE_THRESHOLD}).')

def _set_rows(mask, value, source, needs_review=None):
    _df.loc[mask, 'Value']  = value
    _df.loc[mask, 'Source'] = source
    if needs_review is not None and 'Needs_Review' in _df.columns:
        _df.loc[mask, 'Needs_Review'] = needs_review

_new_rows = []
def _append_row(**kw):
    _new_rows.append({c: kw.get(c) for c in _df.columns})

for e in _entries:
    info = _img_dups.get((e['patent_id'], e['fig_num']))
    if not info: continue
    pid, fname = e['patent_id'], e['filename']
    for field, val in (('dupOfPatent', info['dup_of_patent']), ('dupOfFig', info['dup_of_fig'])):
        mask = ((_df['Patent_ID'].astype(str) == pid) & (_df['Section'] == 'T2')
                & (_df['Sub_Dimension'].astype(str) == f'Image: {fname}') & (_df['Field'] == field))
        if mask.any():
            _set_rows(mask, val, 'siglip')
        else:
            _append_row(Patent_ID=pid, Section='T2', Sub_Dimension=f'Image: {fname}',
                        Field=field, Value=val, Source='siglip', Image_Path=e['image_path'])

# -- Patent-level text-duplicate detection (PatentSBERTa, text-to-text) --
def _t1_val(pid, field):
    m = ((_df['Patent_ID'].astype(str) == pid) & (_df['Section'] == 'T1') & (_df['Field'] == field))
    if m.any():
        v = _df.loc[m, 'Value'].iloc[0]
        return '' if pd.isna(v) else str(v)
    return ''

_pids = _df['Patent_ID'].astype(str).unique().tolist()
_text_by_patent = {pid: (_t1_val(pid, 'title') + '. ' + _t1_val(pid, 'abstract')).strip('. ')
                   for pid in _pids}
_text_dups = detect_text_duplicates(_text_by_patent, sbert, threshold=TEXT_MATCH_THRESHOLD)
print(f'  {len(_text_dups)} text-duplicate patent(s) found (threshold {TEXT_MATCH_THRESHOLD}).')

def _set_t1(pid, field, value, section='T1', sub=None):
    mask = ((_df['Patent_ID'].astype(str) == pid) & (_df['Field'] == field))
    if mask.any():
        _set_rows(mask, value, 'sbert', needs_review=True)
    else:
        _append_row(Patent_ID=pid, Section=section, Sub_Dimension=sub or f'T1 - {field}',
                    Field=field, Value=value, Source='sbert', Needs_Review=True)

for pid, info in _text_dups.items():
    _set_t1(pid, 'isDuplicate', True)
    _set_t1(pid, 'duplicateId', info['dup_of_patent'])
    # duplicateType '1' == 'Same Aircraft' in the HTML wizard's DUP_TYPES.
    _set_t1(pid, 'duplicateType', '1', section='META', sub='Review Metadata')

if _new_rows:
    _df = pd.concat([_df, pd.DataFrame(_new_rows)], ignore_index=True)

with pd.ExcelWriter(_labels_xlsx, engine='openpyxl') as _xl:
    _df.to_excel(_xl, sheet_name='Review', index=False)
print(f'Updated {_labels_xlsx.name}: {len(_img_dups)} image-dup + {len(_text_dups)} text-dup flag(s) written.')


In [ ]:
# ── Flag crops still needing attention (live, from crops_mapping.csv) ──────
# Purely informational: still process every crop normally below, this just
# surfaces which ones need a human look. Reads the SAME live crops_mapping.csv
# (needs_review + crop_quality columns) that src/reviewer.py's review_flags
# mechanism now bakes into each T3_images entry and into source_patents.xlsx —
# so this print always agrees with what ends up in the spreadsheet.
# (Not needs_human_review.csv: that's a one-time pre-review snapshot that goes
# stale immediately and never gains crop_quality at all — see reviewer.py's
# _load_review_flags() docstring.)
import pandas as pd

# Same flat-vs-per-batch fallback as the SKIP_QUALITIES cell above, so this
# print always agrees with what review_flags actually used during the run.
_batch_crops_csv = Path(cfg["paths"]["data_matched"]) / sheet_name / f"crops_mapping_{sheet_name}.csv"
_flat_crops_csv  = Path(cfg["paths"]["data"]) / "crops_mapping.csv"
_crops_csv = _batch_crops_csv if _batch_crops_csv.exists() else _flat_crops_csv

if _crops_csv.exists():
    _crops_df = pd.read_csv(_crops_csv)
    if "crop_quality" not in _crops_df.columns:
        _crops_df["crop_quality"] = ""
    _crops_df["crop_quality"] = _crops_df["crop_quality"].fillna("")
    _needs_attention = (_crops_df["needs_review"] == True) | (_crops_df["crop_quality"] != "")
    _flagged_df = _crops_df.loc[_needs_attention]
else:
    _flagged_df = pd.DataFrame(columns=["patent_id", "output"])

print(f"{len(_flagged_df)} crop(s) in this batch still need attention "
      f"(from {_crops_csv}).")

if not _flagged_df.empty and not summary_df.empty:
    per_patent_flagged = (
        _flagged_df.groupby("patent_id")["output"]
        .apply(list)
        .rename("crops_needing_attention")
    )
    flagged_summary = summary_df.set_index("patent_id").join(per_patent_flagged, how="left")
    flagged_summary["crops_needing_attention"] = flagged_summary["crops_needing_attention"].apply(
        lambda v: v if isinstance(v, list) else []
    )
    flagged_summary["n_needing_attention"] = flagged_summary["crops_needing_attention"].apply(len)
    display(flagged_summary[flagged_summary["n_needing_attention"] > 0]
            [["n_needing_attention", "crops_needing_attention"]])


In [ ]:
# ── Single-patent diagnostic — per-figure match table ───────────────────────
# Set INSPECT_ID to a bare patent_id from this batch to see its T3_images
# table. process_patent() returns its record in-memory only —
# source_patents.xlsx is the persisted output. This calls process_patent()
# live, same as the diagnostic cell near the top of the notebook, just
# with a fuller per-figure table.

INSPECT_ID = None   # e.g. "EP3770063A1"

if INSPECT_ID:
    from src.reviewer import process_patent
    d    = process_patent(
        INSPECT_ID, cfg, excel_index, matched_dir,
        sbert_model=sbert, siglip_bundle=siglip_bundle, skip_siglip=not USE_SIGLIP,
    )
    figs = d.get("T3_images", [])
    icons = {
        "matched":        "🟢",
        "semantic":       "🔵",
        "positional":     "🟡",
        "unmatched":      "🔴",
        "human_required": "⚫",
    }
    print(f"\n{INSPECT_ID}  |  splits={d.get('has_splits')}")
    print(f"{'FILE':<32} {'STATUS':<16} {'CONF':>5}  {'T2_PER':<20}  DESC[:50]")
    print("-" * 110)
    for f in figs:
        icon = icons.get(f.get("match_status", ""), "❓")
        t2   = f.get("T2_predictions") or {}
        per  = (t2.get("per") or {}).get("value", "—")
        desc = (f.get("matched_description") or "")[:50]
        conf = f.get("composite_confidence") or f.get("match_confidence") or 0.0
        print(
            f"{f['file']:<32} {icon} {f.get('match_status',''):<14} "
            f"{float(conf):>5.2f}  {per:<20}  {desc}"
        )


---
## Finalize this batch

Run after you've finished human review for this batch in
`UI_for_taxonomy_caracterization_10.0.html`: load `ml_predict_labels_<batch>.xlsx`
via the "Load Batch" xlsx input, review each patent, then click "Export batch"
(or let it auto-export on the last patent). That downloads
`reviewed_patents_<sheet_name>.xlsx` to your browser's download folder
(`cfg["paths"]["html_review_exports"]`, default `~/Downloads`) — the cell below
reads approval status (`T1 → isApproved`) directly from that file, no manual
`batches.xlsx` editing required.

In [ ]:
# ── Copy approved files → reviewed/{company}/{prototype}/{patent_id}/ ──────
# Run this after you have exported reviewed_patents_<sheet_name>.xlsx from the
# HTML wizard (UI_for_taxonomy_caracterization_10.0.html, "Export batch").
# Approval status comes from that file's T1/isApproved field per patent — NOT
# from batches.xlsx's human_review_status (no manual editing needed there
# anymore). company_canonical/prototype_label still come from batches.xlsx,
# since the wizard export doesn't carry that grouping metadata.
import shutil
from pathlib import Path
import pandas as pd

REVIEWED_XLSX = Path(cfg["paths"]["html_review_exports"]) / f"reviewed_patents_{sheet_name}.xlsx"
if not REVIEWED_XLSX.exists():
    raise FileNotFoundError(
        f"{REVIEWED_XLSX} not found. Export it from the wizard (\"Export batch\" "
        f"button) and make sure it lands in {cfg['paths']['html_review_exports']} first."
    )

review_rows = pd.read_excel(REVIEWED_XLSX, sheet_name="Review")

def _is_true(v):
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() == "true"

approval_rows = review_rows[(review_rows["Section"] == "T1") & (review_rows["Field"] == "isApproved")]
approved_ids = set(
    approval_rows.loc[approval_rows["Value"].apply(_is_true), "Patent_ID"].astype(str)
)
print(f"{len(approved_ids)} patent(s) marked approved in {REVIEWED_XLSX.name}.")

batches_path  = Path(cfg["paths"]["data"]) / "batches.xlsx"
batch_df_live = pd.read_excel(batches_path, sheet_name=sheet_name, dtype=str)
approved = batch_df_live[batch_df_live["patent_id"].astype(str).isin(approved_ids)]
print(f"{len(approved)}/{len(batch_df_live)} patents in {sheet_name} marked approved (via wizard export).")

reviewed_root = Path(cfg["paths"]["reviewed"])
copied, missing = 0, 0
for _, row in approved.iterrows():
    pid       = str(row["patent_id"]).strip()
    company   = str(row["company_canonical"]).strip()
    prototype = str(row["prototype_label"]).strip()
    src_dir   = matched_dir / pid
    dst_dir   = reviewed_root / company / prototype / pid

    if not src_dir.exists():
        print(f"  ⚠ missing matched/ folder for {pid} — skipped")
        missing += 1
        continue
    dst_dir.parent.mkdir(parents=True, exist_ok=True)
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)
    copied += 1

print(f"\nCopied: {copied}   Missing source: {missing}")


In [ ]:
# ── Collect batch outputs → <sheet_name_lower>/ ────────────────────────────
# Copies this batch's xlsx outputs (run_stage01's source_patents_<batch>.xlsx
# and ml_predict_labels_<batch>.xlsx, both written under data_matched/<sheet_name>/
# — there are no per-patent label JSONs in the current pipeline, see
# reviewer.py's module docstring) + any review HTML exports into a flat
# top-level folder for your own validation/inspection. Purely a convenience
# snapshot — skip once you trust the pipeline and are running all batches
# end-to-end without spot-checking each one individually.
import shutil
from pathlib import Path

BATCH_OUT = Path(cfg["paths"]["base"]) / sheet_name.lower()
BATCH_OUT.mkdir(parents=True, exist_ok=True)

# 1. per-batch xlsx outputs written by run_stage01()
batch_data_dir = Path(cfg["paths"]["data_matched"]) / sheet_name
n_xlsx = 0
for f in batch_data_dir.glob("*.xlsx"):
    shutil.copy2(f, BATCH_OUT / f.name)
    n_xlsx += 1

# 2. review HTML exports for this batch, if any were generated
html_src = Path(cfg["paths"]["data"]) / "review_html" / sheet_name
html_dst = BATCH_OUT / "review_html"
n_html = 0
if html_src.exists():
    shutil.copytree(html_src, html_dst, dirs_exist_ok=True)
    n_html = len(list(html_dst.glob("*")))

print(f"Collected {n_xlsx} xlsx file(s) and {n_html} HTML file(s) → {BATCH_OUT}")
